# EN4554 Assignment 2 — Convolutional Neural Networks

**Name:** Rivikula Fernando
**Index number:** 220162T

**AI tool use:** Claude (Anthropic) was used to help write and debug parts of the code in this notebook.
All code was read, run, and checked by me, and the results and interpretations are my own.

Runtime: Google Colab, NVIDIA T4 GPU, PyTorch.

## Setup

In [1]:
import random, time, math, json, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
print('device:', device, '|', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'no GPU')

SEED = 0
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()
torch.backends.cudnn.benchmark = True  # faster convolutions; runs are seeded but not bit-exact

torch 2.11.0+cu130 | torchvision 0.26.0+cu130
device: cuda | Tesla T4


### Data

CIFAR-10 training set (50k) is split into **45k train / 5k validation** with a fixed seed.
Per-channel mean and std are computed on the 45k training images only and applied to both splits.
The whole dataset is kept on the GPU as tensors, which is much faster than a `DataLoader` for 32×32 images.
The **test set is not loaded here**; it is used only once, in Task 4.

In [3]:
raw = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
X_all = torch.tensor(raw.data).permute(0, 3, 1, 2).float().div(255)  # (50000, 3, 32, 32) in [0, 1]
Y_all = torch.tensor(raw.targets)
CLASSES = raw.classes

perm = torch.randperm(len(X_all), generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = perm[:45000], perm[45000:]

MEAN = X_all[train_idx].mean(dim=(0, 2, 3))
STD = X_all[train_idx].std(dim=(0, 2, 3))
normalize = lambda x: (x - MEAN.view(1, 3, 1, 1)) / STD.view(1, 3, 1, 1)

X_train, Y_train = normalize(X_all[train_idx]).to(device), Y_all[train_idx].to(device)
X_val, Y_val = normalize(X_all[val_idx]).to(device), Y_all[val_idx].to(device)
del raw, X_all, Y_all

print('train', tuple(X_train.shape), '| val', tuple(X_val.shape))
print('mean', MEAN.numpy().round(4), '| std', STD.numpy().round(4))
print('val class counts', torch.bincount(Y_val.cpu()).tolist())

100%|██████████| 170M/170M [28:49<00:00, 98.6kB/s] 


train (45000, 3, 32, 32) | val (5000, 3, 32, 32)
mean [0.4912 0.482  0.4464] | std [0.2472 0.2436 0.2617]
val class counts [482, 508, 494, 504, 532, 507, 502, 506, 498, 467]


In [4]:
def batches(X, Y, batch_size, shuffle=True):
    """Yield (x, y) mini-batches from tensors that already live on the device."""
    n = len(X)
    idx = torch.randperm(n, device=X.device) if shuffle else torch.arange(n, device=X.device)
    for i in range(0, n, batch_size):
        j = idx[i:i + batch_size]
        yield X[j], Y[j]

## Task 1: Backpropagation

### (a) Layers with explicit forward and backward passes

Each layer caches what it needs during `forward` and returns $\partial L/\partial(\text{input})$ from `backward`.
No autograd is used: all tensors are plain tensors with `requires_grad=False`.

For a mini-batch $X \in \mathbb{R}^{N\times d_{in}}$ and upstream gradient $G = \partial L/\partial Y$:

| Layer | Forward | Backward |
|---|---|---|
| Linear | $Y = XW^\top + b$ | $\frac{\partial L}{\partial W} = G^\top X,\quad \frac{\partial L}{\partial b} = \sum_n G_n,\quad \frac{\partial L}{\partial X} = G W$ |
| ReLU | $Y = \max(0, X)$ | $\frac{\partial L}{\partial X} = G \odot \mathbb{1}[X > 0]$ |
| Sigmoid | $Y = \sigma(X)$ | $\frac{\partial L}{\partial X} = G \odot Y \odot (1 - Y)$ |
| Softmax + cross-entropy | $p = \mathrm{softmax}(z),\; L = -\frac{1}{N}\sum_n \log p_{n,y_n}$ | $\frac{\partial L}{\partial z} = \frac{1}{N}(p - \mathrm{onehot}(y))$ |

$W$ is stored as (out, in), the same layout as `nn.Linear`, so weights can be copied across for the autograd check in 1(b).
Softmax subtracts the row maximum before exponentiating for numerical stability (this does not change the result).
Sigmoid is included here because it is needed in 1(d).

In [5]:
class Linear:
    def __init__(self, in_features, out_features, init='kaiming', device=device, dtype=torch.float32):
        # kaiming: std = sqrt(2 / fan_in) (for ReLU); xavier: std = sqrt(1 / fan_in) (for sigmoid/tanh)
        gain = 2.0 if init == 'kaiming' else 1.0
        self.W = torch.randn(out_features, in_features, device=device, dtype=dtype) * math.sqrt(gain / in_features)
        self.b = torch.zeros(out_features, device=device, dtype=dtype)
        self.dW = torch.zeros_like(self.W)
        self.db = torch.zeros_like(self.b)

    def forward(self, x):
        self.x = x
        return x @ self.W.T + self.b

    def backward(self, grad_out):
        self.dW = grad_out.T @ self.x
        self.db = grad_out.sum(dim=0)
        return grad_out @ self.W

    def params_and_grads(self):
        return [(self.W, self.dW), (self.b, self.db)]


class ReLU:
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask

    def backward(self, grad_out):
        return grad_out * self.mask

    def params_and_grads(self):
        return []


class Sigmoid:
    def forward(self, x):
        self.y = torch.sigmoid(x)
        return self.y

    def backward(self, grad_out):
        return grad_out * self.y * (1 - self.y)

    def params_and_grads(self):
        return []


class SoftmaxCrossEntropy:
    """Mean softmax cross-entropy over the batch; takes raw logits and integer labels."""
    def forward(self, logits, y):
        z = logits - logits.max(dim=1, keepdim=True).values
        log_p = z - z.exp().sum(dim=1, keepdim=True).log()
        self.p = log_p.exp()
        self.y = y
        return -log_p[torch.arange(len(y)), y].mean()

    def backward(self):
        grad = self.p.clone()
        grad[torch.arange(len(self.y)), self.y] -= 1
        return grad / len(self.y)


class MLP:
    """Stack of Linear layers with an activation between them (none after the last layer)."""
    def __init__(self, sizes, activation='relu', device=device, dtype=torch.float32):
        act_cls = {'relu': ReLU, 'sigmoid': Sigmoid}[activation]
        init = 'kaiming' if activation == 'relu' else 'xavier'
        self.layers = []
        for i, (d_in, d_out) in enumerate(zip(sizes[:-1], sizes[1:])):
            self.layers.append(Linear(d_in, d_out, init=init, device=device, dtype=dtype))
            if i < len(sizes) - 2:
                self.layers.append(act_cls())
        self.loss_fn = SoftmaxCrossEntropy()

    def forward(self, x):
        x = x.flatten(1)
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def loss(self, x, y):
        return self.loss_fn.forward(self.forward(x), y)

    def backward(self):
        grad = self.loss_fn.backward()
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad

    def linear_layers(self):
        return [l for l in self.layers if isinstance(l, Linear)]

    def params_and_grads(self):
        return [pg for l in self.layers for pg in l.params_and_grads()]

Quick sanity check. At initialization the network has learned nothing, so the loss should be of the order of
$\ln 10 \approx 2.303$ (the loss of a uniform guess). It is somewhat higher because Kaiming initialization gives logits
with standard deviation $\approx 1.4$ rather than 0, so predictions are random but not uniform.
Each row of $\partial L/\partial z$ must sum to zero, because the softmax probabilities sum to one.

In [ ]:
set_seed()
mlp = MLP([3 * 32 * 32, 256, 128, 10])
xb, yb = X_train[:256], Y_train[:256]
loss = mlp.loss(xb, yb)
mlp.backward()

print(f'initial loss {loss.item():.4f}  (ln 10 = {math.log(10):.4f}) | logit std {mlp.forward(xb).std().item():.3f}')
print('logit-gradient row sums ~ 0:', mlp.loss_fn.backward().sum(dim=1).abs().max().item())
for i, l in enumerate(mlp.linear_layers()):
    print(f'Linear {i}: W {tuple(l.W.shape)}  dW {tuple(l.dW.shape)}  db {tuple(l.db.shape)}')
print('number of parameters:', sum(p.numel() for p, _ in mlp.params_and_grads()))

### (b) Gradient check

The manual gradients are compared against two independent references:

1. **Autograd.** The same network is rebuilt from PyTorch ops (`F.linear`, `F.relu`/`torch.sigmoid`, `F.cross_entropy`)
   on copies of the same parameters, and `loss.backward()` gives the reference gradients. This also checks the
   gradient w.r.t. the input and the loss value itself.
2. **Centred finite differences.** For a parameter $\theta_i$,
   $\displaystyle \frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta_i + h) - L(\theta_i - h)}{2h}$,
   whose truncation error is $O(h^2)$. Each estimate needs two forward passes, so for the large first weight matrix
   a random subset of 200 entries is checked; every entry of the smaller tensors is checked.

The relative error is $\dfrac{\lVert g_{\text{manual}} - g_{\text{ref}} \rVert_2}{\lVert g_{\text{manual}} \rVert_2 + \lVert g_{\text{ref}} \rVert_2}$.
The check runs in **float64**: in float32 the round-off error of the finite difference, about $\epsilon_{\text{mach}}/h$,
would be as large as the quantity being measured. With $h = 10^{-6}$ in float64 both error sources are around $10^{-10}$.
The network is small ($3072 \to 64 \to 32 \to 10$) and the batch is 16 real CIFAR-10 training images; both ReLU and sigmoid are checked.

In [9]:
import pandas as pd

def rel_error(a, b):
    return ((a - b).norm() / (a.norm() + b.norm()).clamp_min(1e-30)).item()

def autograd_reference(mlp, x, y):
    """Rebuild the network from torch ops on leaf copies of its parameters and let autograd differentiate it."""
    params = [p.detach().clone().requires_grad_(True) for p, _ in mlp.params_and_grads()]
    x = x.detach().clone().requires_grad_(True)
    h, k = x.flatten(1), 0
    for layer in mlp.layers:
        if isinstance(layer, Linear):
            h = F.linear(h, params[k], params[k + 1]); k += 2
        elif isinstance(layer, ReLU):
            h = F.relu(h)
        else:
            h = torch.sigmoid(h)
    loss = F.cross_entropy(h, y)
    loss.backward()
    return loss.item(), [p.grad for p in params], x.grad.flatten(1)

@torch.no_grad()
def finite_difference(mlp, x, y, param, idx, h=1e-6):
    """Centred finite-difference estimate of dL/dparam at the flat indices idx (param is perturbed in place, then restored)."""
    flat = param.view(-1)
    g = torch.empty(len(idx), dtype=param.dtype, device=param.device)
    for n, i in enumerate(idx.tolist()):
        old = flat[i].item()
        flat[i] = old + h; loss_plus = mlp.loss(x, y).item()
        flat[i] = old - h; loss_minus = mlp.loss(x, y).item()
        flat[i] = old
        g[n] = (loss_plus - loss_minus) / (2 * h)
    return g

def gradient_check(activation, n_fd=200, batch=16):
    set_seed()
    mlp = MLP([3 * 32 * 32, 64, 32, 10], activation=activation, dtype=torch.float64)
    x, y = X_train[:batch].double(), Y_train[:batch]

    loss = mlp.loss(x, y).item()
    dx = mlp.backward()
    manual = [g.clone() for _, g in mlp.params_and_grads()]
    ref_loss, ref_grads, ref_dx = autograd_reference(mlp, x, y)

    rows = [dict(activation=activation, tensor='loss value', shape='()',
                 rel_err_autograd=abs(loss - ref_loss) / abs(ref_loss),
                 allclose_autograd=torch.allclose(torch.tensor(loss), torch.tensor(ref_loss)))]
    rows.append(dict(activation=activation, tensor='dL/dx (input)', shape=str(tuple(dx.shape)),
                     rel_err_autograd=rel_error(dx, ref_dx), allclose_autograd=torch.allclose(dx, ref_dx)))
    names = [f'Linear{i}.{n}' for i in range(len(mlp.linear_layers())) for n in ('W', 'b')]
    for name, (p, _), g, g_ref in zip(names, mlp.params_and_grads(), manual, ref_grads):
        n = p.numel()
        idx = torch.arange(n) if n <= n_fd else torch.randperm(n, generator=torch.Generator().manual_seed(SEED))[:n_fd]
        g_fd = finite_difference(mlp, x, y, p, idx)
        g_sub = g.view(-1)[idx.to(g.device)]
        rows.append(dict(activation=activation, tensor=name, shape=str(tuple(p.shape)),
                         rel_err_autograd=rel_error(g, g_ref), allclose_autograd=torch.allclose(g, g_ref),
                         fd_entries_checked=len(idx),
                         rel_err_finite_diff=rel_error(g_sub, g_fd), allclose_finite_diff=torch.allclose(g_sub, g_fd)))
    return rows

t0 = time.time()
check = pd.DataFrame(gradient_check('relu') + gradient_check('sigmoid'))
check['fd_entries_checked'] = check['fd_entries_checked'].astype('Int64')
print(f'gradient check took {time.time() - t0:.1f} s')
with pd.option_context('display.float_format', '{:.2e}'.format):
    display(check)

gradient check took 1.8 s


,activation,tensor,shape,rel_err_autograd,allclose_autograd,fd_entries_checked,rel_err_finite_diff,allclose_finite_diff
0,relu,loss value,(),0.00e+00,True,<NA>,NaN,NaN
1,relu,dL/dx (input),"(16, 3072)",0.00e+00,True,<NA>,NaN,NaN
2,relu,Linear0.W,"(64, 3072)",0.00e+00,True,200,2.11e-09,True
3,relu,Linear0.b,"(64,)",0.00e+00,True,64,2.56e-09,True
4,relu,Linear1.W,"(32, 64)",0.00e+00,True,200,1.47e-09,True
5,relu,Linear1.b,"(32,)",0.00e+00,True,32,1.34e-09,True
6,relu,Linear2.W,"(10, 32)",0.00e+00,True,200,9.07e-10,True
7,relu,Linear2.b,"(10,)",0.00e+00,True,10,9.34e-10,True
8,sigmoid,loss value,(),0.00e+00,True,<NA>,NaN,NaN
9,sigmoid,dL/dx (input),"(16, 3072)",1.48e-16,True,<NA>,NaN,NaN


In [10]:
print('all manual gradients allclose to autograd:        ', bool(check['allclose_autograd'].all()))
print('all manual gradients allclose to finite difference:', bool(check['allclose_finite_diff'].dropna().astype(bool).all()))
print(f"largest relative error vs autograd:          {check['rel_err_autograd'].max():.2e}")
print(f"largest relative error vs finite differences: {check['rel_err_finite_diff'].max():.2e}")

all manual gradients allclose to autograd:         True
all manual gradients allclose to finite difference: True
largest relative error vs autograd:          1.48e-16
largest relative error vs finite differences: 7.73e-08


**Interpretation.** Against autograd, every relative error is at the level of float64 machine precision
($\sim 10^{-16}$) and every `torch.allclose` check passes: the manual backward pass computes the same gradients as
PyTorch, for the loss value, the input gradient and every weight and bias, with both activations.
Against centred finite differences the relative errors are larger but still tiny (roughly $10^{-9}$ to $10^{-7}$)
and all `torch.allclose` checks pass. These errors are not a sign of a bug: they are the expected error of the
finite-difference estimate itself, a mix of $O(h^2)$ truncation and $O(\epsilon_{\text{mach}}/h)$ round-off. A wrong
backward formula would instead give relative errors of order $10^{-2}$ to $1$.
The largest finite-difference errors appear in the early layers of the sigmoid network: there the gradients are
small (sigmoid's derivative is at most 0.25 per layer, see 1(d)), so the same absolute estimation error is a larger
fraction of the gradient. With ReLU, a finite difference could in principle be wrong if a pre-activation lay within
$h$ of the kink at 0, but with $h = 10^{-6}$ this did not happen.